# 01 · The experiment and data quality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/academic-research/time-preference-choice-models/notebooks/01_experiment_and_data_quality.ipynb)

Why do people take a smaller reward now over a larger one later, and can that be shaped? This
notebook sets up a simulated online experiment that asks exactly that, and cleans its data with
rules fixed before any model is fitted.

Each participant makes 60 choices between a smaller-sooner and a larger-later amount, plus 4 catch
trials where the later option is smaller, so choosing it means the person was not reading. Half the
participants see delays ("in 30 days"), half see calendar dates ("on 21 December").

All data are simulated. Participant codes are random, and no personal data exist anywhere in the
project. Because the generating process is known, every later step can be checked against the truth.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "academic-research/time-preference-choice-models"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "timepref").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: time-preference-choice-models


In [2]:
from timepref import config as C, data, quality as Q, models as M, evaluate as E, effects as F, nlp as N, reporting as R

if not (Path("data") / "trials.csv.gz").exists():
    data.save(data.generate(), Path("data"))
tables = data.load(Path("data"))
clean, quality_log, rules = Q.apply(tables)
print(f"{clean['pid'].nunique()} participants, {len(clean):,} choices after exclusions")

399 participants, 23,940 choices after exclusions


## What a participant sees

Each row is one choice. `choice_ll` is 1 when the person took the larger-later option.

In [3]:
tables["trials"].query("submission == 1").head(8)[
    ["pid", "trial", "frame", "ss_amount", "ss_delay", "ll_amount", "ll_delay", "catch", "choice_ll", "rt_ms"]]

,pid,trial,frame,ss_amount,ss_delay,ll_amount,ll_delay,catch,choice_ll,rt_ms
0,PMW3892,1,delay,5.0,7.0,6.0,14.0,False,0,1613
1,PMW3892,2,delay,37.0,30.0,55.0,90.0,False,0,1659
2,PMW3892,3,delay,31.0,0.0,62.0,30.0,False,0,1294
3,PMW3892,4,delay,62.0,0.0,57.0,90.0,True,0,1229
4,PMW3892,5,delay,16.0,0.0,37.0,90.0,False,0,1997
5,PMW3892,6,delay,8.0,0.0,9.0,60.0,False,0,1566
6,PMW3892,7,delay,44.0,0.0,91.0,14.0,False,0,1282
7,PMW3892,8,delay,32.0,0.0,75.0,180.0,False,0,1130


## Who is in the simulated population

Four decision types, so the models can be scored against a known truth: hyperbolic, quasi-hyperbolic
and exponential discounters, and people who use the ITCH heuristic (Ericson et al. 2015) and never
discount at all. Sixteen careless responders and twelve repeat submissions were added on purpose.
A real study has no such table.

In [4]:
tables["truth"]["type"].value_counts()

type
hyperbolic          135
quasi_hyperbolic    112
itch                 91
exponential          62
careless             16
Name: count, dtype: int64

## Pre-registered exclusion rules

The rules are fixed before any model sees the data. Both content rules are needed: the speed rule
catches every careless responder, and the catch-trial rule catches 14 of 16 plus one genuine
participant who failed two catch trials.

In [5]:
quality_log

,rule,check,participants,detail
0,Q1,Duplicate submissions (second attempt dropped),12,12 codes submitted twice
1,Q2,Failed 2 or more of 4 catch trials,15,"distribution of fails: {0: 357, 1: 44, 2: 3, 3: 4, 4: 8}"
2,Q3,Median response time under 450 ms,16,fastest median 294 ms
3,Q4,Same answer on all 60 real trials,7,flag only
4,Q5,Conditions balanced after exclusions,0,"delay/date before 207/209, after 200/199, p = 1.00"


In [6]:
Q.flow(tables, rules)

,step,n
0,Submissions received,428
1,Unique participant codes,416
2,Excluded: catch trials only,1
3,Excluded: too fast only,2
4,Excluded: both,14
5,Analysed,399


In [7]:
excluded = set(rules[1].flagged) | set(rules[2].flagged)
careless = set(tables["truth"].query("careless != 'no'")["pid"])
print("careless responders caught:", len(careless & excluded), "of", len(careless))
print("genuine participants excluded:", len(excluded - careless))

careless responders caught: 16 of 16
genuine participants excluded: 1


## A first look at the choices

Patience rises with the size of the premium for waiting and falls with the wait itself. The date
frame already shows a higher share of later choices before any model is fitted.

In [8]:
c = clean.assign(premium=pd.cut(clean["ll_amount"] / clean["ss_amount"] - 1, [0, 0.25, 0.5, 1, 1.5],
                                  labels=["up to 25%", "25-50%", "50-100%", "100-150%"]),
              wait=pd.cut(clean["ll_delay"] - clean["ss_delay"], [0, 14, 60, 180], labels=["1-2 weeks", "3-8 weeks", "3-6 months"]))
(c.pivot_table(index="premium", columns=["frame", "wait"], values="choice_ll", aggfunc="mean", observed=True) * 100).round(0)

frame          date                          delay                     
wait      1-2 weeks 3-8 weeks 3-6 months 1-2 weeks 3-8 weeks 3-6 months
premium                                                                
up to 25%      60.0      38.0       22.0      45.0      23.0       10.0
25-50%         83.0      68.0       45.0      70.0      49.0       21.0
50-100%        93.0      85.0       67.0      86.0      70.0       44.0
100-150%       97.0      94.0       82.0      93.0      87.0       67.0